In [5]:
import argparse
from datetime import datetime, timezone
import pandas as pd

from aind_data_schema.components.identifiers import Code, DataAsset
from aind_data_schema.core.processing import (
    DataProcess,
    Processing,
    ProcessName,
    ProcessStage,
    ResourceTimestamped,
    ResourceUsage,
)
from aind_data_schema_models.units import MemoryUnit
from aind_data_schema_models.system_architecture import OperatingSystem, CPUArchitecture
from codeocean import CodeOcean
import os
os.sys.path.append('/root/capsule/code/beh_ephys_analysis')
from utils.beh_functions import parseSessionID
import json

# Generating two histology-related data asssets' procesing metadata

In [6]:
client = CodeOcean(domain="https://codeocean.allenneuraldynamics.org", token=os.getenv("API_SECRET"))

In [7]:
meta_data_file = '/root/capsule/code/data_management/meta_data_processing.xlsx'
session_meta = pd.read_excel(meta_data_file, sheet_name='sessions')
stan_meta = pd.read_excel(meta_data_file, sheet_name='animal_stan')
glm_meta = pd.read_excel(meta_data_file, sheet_name='animal_glm')
ccf_meta = pd.read_excel(meta_data_file, sheet_name='all_ccf')
ccf_meta = ccf_meta.dropna(subset=['url'])
dorsal_edge_meta = pd.read_excel(meta_data_file, sheet_name='all_dorsal_edge')

computation_params_file = '/root/capsule/code/data_management/processing_params.json'
with open(computation_params_file, 'r') as f:
    computation_params = json.load(f)

## CCF for neurons

In [8]:
ccf_data_asset_id = 'c1a35fd0-c3aa-47a8-ba40-288b1e39a86a'
ts = client.data_assets.get_data_asset(data_asset_id=ccf_data_asset_id).created
t = datetime.fromtimestamp(ts, tz=timezone.utc)

In [9]:
data_file = '/root/capsule/code/data_management/LC-NE_probe_annotation.xlsx'
data_asset_names = ['ibl_converted_ccf_fix', 'raw_smartspim', 'stitched_smartspim']
data_asset_ids = pd.read_excel(data_file, sheet_name='LC-NE_probe_annotation')[data_asset_names].dropna().values.flatten().tolist()
data_asset_names_full = [client.data_assets.get_data_asset(data_asset_id=id).name  for id in data_asset_ids]
data_assets = [DataAsset(url=name) for name in data_asset_names_full]

In [10]:
dp_list = []
for row_ind, row in ccf_meta.iterrows():
    curr_code = Code(
        url=row['url'],
        run_script=row['run_script'],
        version=row['version'],
        input_data=data_assets,
    )

    curr_dp = DataProcess(
                process_type=ProcessName.OTHER,
                name=row['name'],
                experimenters=["Sue Su"],
                stage=ProcessStage.ANALYSIS,
                start_date_time=t,
                output_path=row['output'],
                code=curr_code,
                notes=row['name'] + '. ' + row['additional_note'] if pd.notna(row['additional_note']) else row['name'],
                )
    dp_list.append(curr_dp)
p_ccf_meta = Processing.create_with_sequential_process_graph(
    data_processes=dp_list)

## CCF for LC dorsal edges

In [11]:
dorsal_edge_data_asset_id = 'ac7c7961-9178-4bf9-9d66-0a426cf3cc24'
ts = client.data_assets.get_data_asset(data_asset_id=dorsal_edge_data_asset_id).created
t = datetime.fromtimestamp(ts, tz=timezone.utc)
data_file = '/root/capsule/code/data_management/LC-NE_probe_annotation.xlsx'
data_asset_names = ['stitched_smartspim']
data_asset_ids = pd.read_excel(data_file, sheet_name='LC-NE_probe_annotation')[data_asset_names].dropna().values.flatten().tolist()
data_asset_names_full = [client.data_assets.get_data_asset(data_asset_id=id).name  for id in data_asset_ids]
data_assets = [DataAsset(url=name) for name in data_asset_names_full]

In [12]:
dp_list = []
for row_ind, row in dorsal_edge_meta.iterrows():
    curr_code = Code(
        url=row['url'],
        run_script=row['run_script'],
        version=row['version'],
        input_data=data_assets,
    )

    curr_dp = DataProcess(
                process_type=ProcessName.OTHER,
                name=row['name'],
                experimenters=["Sue Su"],
                stage=ProcessStage.ANALYSIS,
                start_date_time=t,
                output_path=row['output'],
                code=curr_code,
                notes=row['name'] + '. ' + row['additional_note'] if pd.notna(row['additional_note']) else row['name'],
                )
    dp_list.append(curr_dp)
p_dorsal_edge_meta = Processing.create_with_sequential_process_graph(
    data_processes=dp_list)